In [1]:
import pandas as pd
import os

In [ ]:
"""
Final Columns for Data Load
TABLE fact_gen(
  date_id INT,
  state_id INT,
  fuel_id INT,
  generation DECIMAL(12, 2)

TABLE fact_prices(
  date_id INT,
  state_id INT,
  sector_id INT,
  price_per_kwh DECIMAL(10,4),

TABLE fact_fuel_prices(
  date_id INT,
  state_id INT,
  fuel_id INT,
  price_per_mmbtu DECIMAL(10,4),

"""

In [2]:
gen_df = pd.read_csv("../data/gen_data.csv")
gen_df.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2025-03,90,Pacific,1,Electric Utility,ALL,all fuels,17842.73950,thousand megawatthours
1,2025-03,90,Pacific,1,Electric Utility,AOR,all renewables,822.00528,thousand megawatthours
2,2025-03,90,Pacific,1,Electric Utility,BIO,biomass,37.41079,thousand megawatthours
3,2025-03,90,Pacific,1,Electric Utility,COL,"coal, excluding waste coal",36.42772,thousand megawatthours
4,2025-03,90,Pacific,1,Electric Utility,COW,all coal products,36.42772,thousand megawatthours


In [3]:
price_df = pd.read_csv("../data/price_data.csv")
price_df.head()

,period,stateid,stateDescription,sectorid,sectorName,price,price-units
0,2025-03,AK,Alaska,ALL,all sectors,22.96,cents per kilowatt-hour
1,2025-03,AK,Alaska,COM,commercial,22.10,cents per kilowatt-hour
2,2025-03,AK,Alaska,IND,industrial,20.17,cents per kilowatt-hour
3,2025-03,AK,Alaska,OTH,other,NaN,cents per kilowatt-hour
4,2025-03,AK,Alaska,RES,residential,25.79,cents per kilowatt-hour


In [4]:
fuel_df = pd.read_csv("../data/fuel_data.csv")
fuel_df.head()

#duoarea is state abrv with a leading s

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2025-03,SAR,USA-AR,EPG0,Natural Gas,PEU,Electric Power Price,N3045AR3,Arkansas Natural Gas Price Sold to Electric Po...,NaN,$/MCF
1,2025-03,SNE,USA-NE,EPG0,Natural Gas,PEU,Electric Power Price,N3045NE3,Nebraska Natural Gas Price Sold to Electric Po...,7.28,$/MCF
2,2025-03,SMI,USA-MI,EPG0,Natural Gas,PEU,Electric Power Price,N3045MI3,Michigan Natural Gas Price Sold to Electric Po...,NaN,$/MCF
3,2025-03,SCO,COLORADO,EPG0,Natural Gas,PEU,Electric Power Price,N3045CO3,Colorado Natural Gas Price Sold to Electric Po...,NaN,$/MCF
4,2025-03,SAK,USA-AK,EPG0,Natural Gas,PEU,Electric Power Price,N3045AK3,Alaska Natural Gas Price Sold to Electric Powe...,8.34,$/MCF


## Dealing with States
- creating maps across df

In [ ]:
#convert columns to dict
#pd.Series(df.Letter.values,index=df.Position).to_dict()
stateShortToLong_map = pd.Series(price_df.stateDescription.values, index=price_df.stateid).to_dict()
stateLongToShort_map = pd.Series(price_df.stateid.values, index=price_df.stateDescription).to_dict()

In [13]:
#NA from Gen Pacific and Puerto Rico
gen_df["stateShort"] = gen_df.stateDescription.map(stateLongToShort_map)

In [18]:
gen_df[["stateDescription", "stateShort"]].drop_duplicates().values

array([['Pacific', nan],
       ['Alaska', 'AK'],
       ['Alabama', 'AL'],
       ['Arkansas', 'AR'],
       ['Arizona', 'AZ'],
       ['California', 'CA'],
       ['Colorado', 'CO'],
       ['Connecticut', 'CT'],
       ['District of Columbia', 'DC'],
       ['Delaware', 'DE'],
       ['East North Central', 'ENC'],
       ['East South Central', 'ESC'],
       ['Florida', 'FL'],
       ['Georgia', 'GA'],
       ['Hawaii', 'HI'],
       ['Iowa', 'IA'],
       ['Idaho', 'ID'],
       ['Illinois', 'IL'],
       ['Indiana', 'IN'],
       ['Kansas', 'KS'],
       ['Kentucky', 'KY'],
       ['Louisiana', 'LA'],
       ['Massachusetts', 'MA'],
       ['Middle Atlantic', 'MAT'],
       ['Maryland', 'MD'],
       ['Maine', 'ME'],
       ['Michigan', 'MI'],
       ['Minnesota', 'MN'],
       ['Missouri', 'MO'],
       ['Mississippi', 'MS'],
       ['Montana', 'MT'],
       ['Mountain', 'MTN'],
       ['North Carolina', 'NC'],
       ['North Dakota', 'ND'],
       ['Nebraska', 'NE'],
       ['Ne

In [21]:
#Price doesn't have data for Pacific Region or Puerto Rico
gen_df.loc[gen_df.stateShort.isna()][["stateDescription", "stateShort"]].drop_duplicates()

,stateDescription,stateShort
0,Pacific,NaN
12783,Puerto Rico,NaN


In [24]:
fuel_df["stateShort"] = fuel_df.duoarea.str[1:]
fuel_df["stateLong"] = fuel_df.stateShort.map(stateShortToLong_map)
fuel_df.head()

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units,stateShort,stateLong
0,2025-03,SAR,USA-AR,EPG0,Natural Gas,PEU,Electric Power Price,N3045AR3,Arkansas Natural Gas Price Sold to Electric Po...,NaN,$/MCF,AR,Arkansas
1,2025-03,SNE,USA-NE,EPG0,Natural Gas,PEU,Electric Power Price,N3045NE3,Nebraska Natural Gas Price Sold to Electric Po...,7.28,$/MCF,NE,Nebraska
2,2025-03,SMI,USA-MI,EPG0,Natural Gas,PEU,Electric Power Price,N3045MI3,Michigan Natural Gas Price Sold to Electric Po...,NaN,$/MCF,MI,Michigan
3,2025-03,SCO,COLORADO,EPG0,Natural Gas,PEU,Electric Power Price,N3045CO3,Colorado Natural Gas Price Sold to Electric Po...,NaN,$/MCF,CO,Colorado
4,2025-03,SAK,USA-AK,EPG0,Natural Gas,PEU,Electric Power Price,N3045AK3,Alaska Natural Gas Price Sold to Electric Powe...,8.34,$/MCF,AK,Alaska


In [25]:
fuel_df[["stateShort", "stateLong"]].drop_duplicates().values

array([['AR', 'Arkansas'],
       ['NE', 'Nebraska'],
       ['MI', 'Michigan'],
       ['CO', 'Colorado'],
       ['AK', 'Alaska'],
       ['NH', 'New Hampshire'],
       ['RI', 'Rhode Island'],
       ['LA', 'Louisiana'],
       ['IL', 'Illinois'],
       ['PA', 'Pennsylvania'],
       ['OH', 'Ohio'],
       ['AZ', 'Arizona'],
       ['IA', 'Iowa'],
       ['KY', 'Kentucky'],
       ['WV', 'West Virginia'],
       ['OR', 'Oregon'],
       ['WA', 'Washington'],
       ['ID', 'Idaho'],
       ['SC', 'South Carolina'],
       ['CT', 'Connecticut'],
       ['MD', 'Maryland'],
       ['WI', 'Wisconsin'],
       ['MT', 'Montana'],
       ['NC', 'North Carolina'],
       ['CA', 'California'],
       ['FL', 'Florida'],
       ['ME', 'Maine'],
       ['VT', 'Vermont'],
       ['MS', 'Mississippi'],
       ['NM', 'New Mexico'],
       ['OK', 'Oklahoma'],
       ['NY', 'New York'],
       ['NV', 'Nevada'],
       ['IN', 'Indiana'],
       ['WY', 'Wyoming'],
       ['UT', 'Utah'],
       ['MA', 

In [ ]:
# DEALING WITH STATES
# lets normalize the columns across the tables

# take stateid and stateDescription from price to create a map
# use duoarea from fuel to add state name

## Dealing with Fuel Types

In [13]:
gen_df["fuelTypeDescription"].unique()

<StringArray>
[                               'all fuels',
                           'all renewables',
                                  'biomass',
               'coal, excluding waste coal',
                        'all coal products',
                      'distillate fuel oil',
                             'fossil fuels',
                               'geothermal',
            'hydro-electric pumped storage',
               'conventional hydroelectric',
                             'landfill gas',
                             'lignite coal',
                   'municiapl landfill gas',
                              'natural gas',
                'natural gas & other gases',
                                  'nuclear',
                              'other gases',
                         'other renewables',
                                    'other',
                        'petroleum liquids',
                                'petroleum',
                                'renewabl

In [14]:
gen_df["fueltypeid"].unique()

<StringArray>
['ALL', 'AOR', 'BIO', 'COL', 'COW', 'DFO', 'FOS', 'GEO', 'HPS', 'HYC', 'LFG',
 'LIG', 'MLG',  'NG', 'NGO', 'NUC', 'OB2', 'OBW', 'OOG', 'ORW', 'OTH', 'PEL',
 'PET', 'REN', 'RFO', 'SPV', 'SUB', 'SUN', 'WAS', 'WND', 'WNT', 'WOC', 'WOO',
 'WWW', 'MSB',  'PC',  'RC', 'STH', 'BIS', 'BIT', 'DPV', 'TPV', 'TSN', 'ANT',
 'WNS']
Length: 45, dtype: str

In [17]:
df=gen_df.drop_duplicates(subset = ["fueltypeid", "fuelTypeDescription"])

In [19]:
df[["fueltypeid", "fuelTypeDescription"]]

,fueltypeid,fuelTypeDescription
0,ALL,all fuels
1,AOR,all renewables
2,BIO,biomass
3,COL,"coal, excluding waste coal"
4,COW,all coal products
5,DFO,distillate fuel oil
6,FOS,fossil fuels
7,GEO,geothermal
8,HPS,hydro-electric pumped storage
9,HYC,conventional hydroelectric


In [5]:
gen_df.dtypes

period                     str
location                   str
stateDescription           str
sectorid                 int64
sectorDescription          str
fueltypeid                 str
fuelTypeDescription        str
generation             float64
generation-units           str
dtype: object

In [ ]:
#fueltypeid = fuel_category
#fuelTypeDescription = fuel_name

In [ ]:
#at stage do I add in the id columns? I need to load those first but I also need to extract the values to create those dim table?